# 03 GDC 데이터 층 가용성과 슬라이드 병리 주석
GDC API로 메타데이터만 조회한다 (파일 다운로드 없음).
1. 환자별 데이터 층 가용성: DX, TS·BS, RNA, 450K 메틸화, miRNA, MAF, CNV, RPPA (원발 종양 표본만)
2. TS·BS·DX 슬라이드 병리 주석: 종양세포 %, 림프구 침윤 % 등 → 28번 조직 불일치 분석에 사용
- 커널: `C:\Users\Admin\EXAONEPath\exaonepath-venv` (인터넷 필요)
- 출력: `C:\Users\Admin\EXAONEPath_data\v4\03_gdc_inventory\{gdc_files.csv, availability.csv, slide_annotations.csv}`

In [1]:
# 1. 경로와 GDC 조회 함수
from pathlib import Path
import requests, pandas as pd

OUT2 = Path(r"C:\Users\Admin\EXAONEPath_data\v4\02_cohort")
OUT  = Path(r"C:\Users\Admin\EXAONEPath_data\v4\03_gdc_inventory")
OUT.mkdir(parents=True, exist_ok=True)
PROJ = ["TCGA-LUAD", "TCGA-LUSC"]
co = pd.read_csv(OUT2 / "cohort.csv").query("excluded.isna() or excluded == ''")

def gdc(endpoint, filters, fields):
    r = requests.post(f"https://api.gdc.cancer.gov/{endpoint}", timeout=600,
                      json={"filters": filters, "fields": ",".join(fields), "format": "JSON", "size": 100000})
    r.raise_for_status()
    d = r.json()["data"]
    assert len(d["hits"]) == d["pagination"]["total"], "페이지 누락"
    return d["hits"]

In [2]:
# 2. 파일 목록 → 환자 × 데이터 층
LAYER = {"Gene Expression Quantification": "RNA", "miRNA Expression Quantification": "miRNA",
         "Masked Somatic Mutation": "MAF", "Gene Level Copy Number": "CNV", "Protein Expression Quantification": "RPPA"}
types = list(LAYER) + ["Slide Image", "Methylation Beta Value"]
hits = gdc("files",
           {"op": "and", "content": [{"op": "in", "content": {"field": "cases.project.project_id", "value": PROJ}},
                                     {"op": "in", "content": {"field": "data_type", "value": types}}]},
           ["file_id", "file_name", "data_type", "experimental_strategy", "platform",
            "cases.submitter_id", "cases.project.project_id", "cases.samples.sample_type"])

f = pd.DataFrame([(h["file_id"], h["file_name"], h["data_type"], h.get("experimental_strategy"), h.get("platform"),
                   c["project"]["project_id"][5:], c["submitter_id"], s.get("sample_type"))
                  for h in hits for c in h["cases"] for s in c.get("samples", [])],
                 columns=["file_id", "file_name", "data_type", "strategy", "platform", "cancer", "patient_id", "sample_type"])
f["layer"] = f.data_type.map(LAYER)
f.loc[f.data_type == "Slide Image", "layer"] = f.strategy.map({"Diagnostic Slide": "DX", "Tissue Slide": "TS_BS"})
f.loc[f.data_type == "Methylation Beta Value", "layer"] = f.platform.str.contains("450").map({True: "Meth450", False: "Meth_other"})
f.to_csv(OUT / "gdc_files.csv", index=False, encoding="utf-8-sig")

pt = f[f.sample_type == "Primary Tumor"]
av = (pd.crosstab([pt.patient_id, pt.cancer], pt.layer) > 0).reset_index()
av["in_cohort"] = av.patient_id.isin(co.patient_id)
av.to_csv(OUT / "availability.csv", index=False, encoding="utf-8-sig")
av.groupby(["cancer", "in_cohort"]).sum(numeric_only=True).astype(int)

layer             CNV   DX  MAF  Meth450  Meth_other  RNA  RPPA  TS_BS  miRNA
cancer in_cohort                                                             
LUAD   False      117   75  161       55         128  114    86    119    112
       True       402  403  398      403          49  403   279    387    401
LUSC   False      157  131  149       23         133  154    90    157    145
       True       347  347  341      347           0  347   238    331    343

In [3]:
# 3. 슬라이드 병리 주석 (cases.samples.portions.slides)
SL = ["submitter_id", "section_location", "percent_tumor_cells", "percent_tumor_nuclei", "percent_normal_cells",
      "percent_stromal_cells", "percent_necrosis", "percent_lymphocyte_infiltration",
      "percent_monocyte_infiltration", "percent_neutrophil_infiltration"]
hits = gdc("cases", {"op": "in", "content": {"field": "project.project_id", "value": PROJ}},
           ["submitter_id", "project.project_id", "samples.sample_type"] + [f"samples.portions.slides.{k}" for k in SL])

ann = pd.DataFrame([{"patient_id": c["submitter_id"], "cancer": c["project"]["project_id"][5:],
                     "sample_type": s.get("sample_type"), **{k: sl.get(k) for k in SL}}
                    for c in hits for s in c.get("samples", []) for p in s.get("portions", []) for sl in p.get("slides", [])])
ann = ann.rename(columns={"submitter_id": "slide_submitter_id"}).drop_duplicates("slide_submitter_id")
ann["slide_type"] = ann.slide_submitter_id.str.extract(r"-(TS|BS|DX|MS)[0-9A-Z]*$", expand=False)
num = [k for k in SL if k.startswith("percent_")]
ann[num] = ann[num].apply(pd.to_numeric, errors="coerce")
ann["in_cohort"] = ann.patient_id.isin(co.patient_id)
ann.to_csv(OUT / "slide_annotations.csv", index=False, encoding="utf-8-sig")

In [4]:
# 4. 요약: 코호트 원발 종양에서 슬라이드 종류별 주석 보유 수
a = ann[ann.in_cohort & (ann.sample_type == "Primary Tumor")]
a.groupby(["cancer", "slide_type"]).agg(slides=("slide_submitter_id", "size"), patients=("patient_id", "nunique"),
                                        tumor=("percent_tumor_cells", "count"),
                                        lymph=("percent_lymphocyte_infiltration", "count"))

slides  patients  tumor  lymph
cancer slide_type                                
LUAD   BS             227       189    219    100
       DX             449       403      0      0
       TS             405       373    392    239
LUSC   BS             146       136    142     58
       DX             360       347      0      0
       TS             324       316    308    206